In [ ]:
# Instalar dependências
!pip install diffusers==0.33.1 transformers accelerate einops gradio ffmpeg-python safetensors --quiet

In [ ]:
# Login na Hugging Face
from huggingface_hub import login
login()

In [ ]:
# Imports e configurações
import os
import torch
from PIL import Image
import gradio as gr
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import export_to_video, load_image
import glob
import traceback
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('outputs', exist_ok=True)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

In [ ]:
# Carregar componentes base
from transformers import logging
logging.set_verbosity_error()
from diffusers import CogVideoXTransformer3DModel, AutoencoderKLCogVideoX
transformer = CogVideoXTransformer3DModel.from_pretrained(
    'THUDM/CogVideoX-5b-I2V',
    subfolder='transformer',
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map='balanced',
    offload_folder='modelo_offload'
)
vae = AutoencoderKLCogVideoX.from_pretrained(
    'THUDM/CogVideoX-5b-I2V',
    subfolder='vae',
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map='balanced',
    offload_folder='modelo_offload'
)

In [ ]:
# Carregar tokenizer, text_encoder e scheduler
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import DDIMScheduler
text_encoder = CLIPTextModel.from_pretrained(
    'openai/clip-vit-large-patch14',
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map='balanced',
    offload_folder='modelo_offload'
)
tokenizer = CLIPTokenizer.from_pretrained('openai/clip-vit-large-patch14')
scheduler = DDIMScheduler.from_pretrained(
    'THUDM/CogVideoX-5b-I2V',
    subfolder='scheduler'
)

In [ ]:
# Montar pipeline
pipe = CogVideoXImageToVideoPipeline(
    transformer=transformer,
    vae=vae,
    tokenizer=tokenizer,
    text_encoder=text_encoder,
    scheduler=scheduler
).to(device)

In [ ]:
# Função final de geração com truncamento manual via embeddingsdef generate_video(image_path, prompt, width, height, frames, steps):    try:        if not os.path.exists(image_path):            raise ValueError('Imagem não encontrada.')        for f in glob.glob('outputs/*.mp4'):            os.remove(f)        image = load_image(image_path).convert('RGB').resize((width, height))        input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)        negative_ids = tokenizer('', truncation=False, padding='max_length', max_length=input_ids.shape[-1], return_tensors='pt').input_ids.to(device)        max_length = tokenizer.model_max_length        concat_embeds = []        neg_embeds = []        for i in range(0, input_ids.shape[-1], max_length):            concat_embeds.append(text_encoder(input_ids[:, i: i + max_length])[0])            neg_embeds.append(text_encoder(negative_ids[:, i: i + max_length])[0])        prompt_embeds = torch.cat(concat_embeds, dim=1)        negative_prompt_embeds = torch.cat(neg_embeds, dim=1)        result = pipe(            image=image,            prompt_embeds=prompt_embeds,            negative_prompt_embeds=negative_prompt_embeds,            guidance_scale=5,            num_inference_steps=steps,            num_frames=frames        )        frames_result = result.frames[0]        out_path = f'outputs/cogvideo_{width}x{height}.mp4'        export_to_video(frames_result, out_path, fps=6)        return out_path, 'Vídeo gerado com sucesso.'    except Exception as e:        print('Erro:', e)        traceback.print_exc()        return None, 'Erro: ' + str(e)

In [ ]:
# Interface Gradio
with gr.Blocks() as demo:
    gr.Markdown('Geração de vídeo a partir de imagem')
    with gr.Row():
        img = gr.Image(type='filepath', label='Imagem')
        prm = gr.Textbox(label='Prompt (ex: ocean waves at sunset)')
    with gr.Row():
        width = gr.Slider(256, 1280, value=720, step=64, label='Largura')
        height = gr.Slider(256, 720, value=480, step=64, label='Altura')
    with gr.Row():
        frames = gr.Slider(2, 16, value=8, step=1, label='Frames')
        steps = gr.Slider(4, 50, value=16, step=1, label='Inference Steps')
    btn = gr.Button('Gerar Vídeo')
    vid = gr.Video(label='Resultado')
    status = gr.Markdown('Pronto para gerar.')
    def wrapper_generate(image_path, prompt, width, height, frames, steps):
        video_path, msg = generate_video(image_path, prompt, width, height, frames, steps)
        return video_path, msg
    btn.click(fn=wrapper_generate, inputs=[img, prm, width, height, frames, steps], outputs=[vid, status])
    demo.launch(share=True, debug=False)